In [1]:
import numpy as np
import altair as alt 
import pandas as pd

/usr/lib/python3/dist-packages/pytz/__init__.py:31: SyntaxWarning: invalid escape sequence '\s'
  match = re.match("^#\s*version\s*([0-9a-z]*)\s*$", line)


In [2]:
def init_parameters(n_features, n_neurons, n_output): 

    np.random.seed(100)
    W1 = np.random.uniform(size = (n_features, n_neurons))
    b1 = np.random.uniform(size = (1, n_neurons))

    W2 = np.random.uniform(size = (n_neurons, n_output))
    b2 = np.random.uniform(size = (1, n_output))

    return {
        "W1" : W1 
        , "b1" : b1 
        , "W2" : W2
        , "b2" : b2 
    }

In [3]:
def linear_function(W, X, b): 
    return (X @ W)+ b 

In [4]:
def sigmoid_func(Z): 
    return 1 / (1 + np.exp(-Z))

In [5]:
def cost_function(A, y): 
    return -np.mean(y * np.log(A) + (1-y) * np.log(1-A))

In [6]:
def predict(X, W1, W2, b1, b2): 
    Z1 = linear_function(W1, X, b1)
    S1 = sigmoid_func(Z1)
    Z2 = linear_function(W2, S1, b2)
    S2 = sigmoid_func(Z2)
    return np.where(S2 >= 0.5, 1, 0)

In [7]:
def fit(X, y, n_features = 2, n_neurons = 3, n_output = 1, iterations = 10, eta = 0.001): 

    params = init_parameters(
        n_features= n_features
        , n_neurons= n_neurons
        , n_output= n_output
    )

    errors = [] 

    for _ in range(iterations): 

        Z1 = linear_function(params['W1'], X, params['b1'])
        S1 = sigmoid_func(Z1)
        Z2 = linear_function(params['W2'], S1, params['b2'])
        S2 = sigmoid_func(Z2)

        error = cost_function(S2, y)
        errors.append(error)

        delta2 =S2 - y
        W2_gradients = S1.T @ delta2 
        params["W2"] = params["W2"] - W2_gradients * eta

        params["b2"] = params["b2"] - np.sum(delta2, axis = 0, keepdims= True) * eta 

        delta1 = (delta2 @ params["W2"].T) * S1 * (1 - S1)
        W1_gradients = X.T @ delta1 
        params["W1"] = params["W1"] - W1_gradients * eta 

        params["b1"] = params["b1"] - np.sum(delta1, axis = 0, keepdims= True) * eta 

    return errors, params 


        

In [8]:
y = np.array([[0, 1, 1, 0]]).T 
X = np.array([[0, 0, 1, 1]
              ,[0, 1, 0, 1]]).T 

In [9]:
errors, params = fit(X, y, iterations=5000, eta = 0.1)

In [10]:
y_pred = predict(X, params["W1"], params["W2"], params["b1"], params["b2"])
num_correct_predictions = (y_pred == y).sum()
accuracy = (num_correct_predictions / y.shape[0]) * 100
print('Multi-layer perceptron accuracy: %.2f%%' % accuracy)


Multi-layer perceptron accuracy: 100.00%


In [11]:
alt.data_transformers.disable_max_rows()
df = pd.DataFrame({"errors":errors, "time-step": np.arange(0, len(errors))})
alt.Chart(df).mark_line().encode(x="time-step", y="errors").properties(title='Chart 2')


alt.Chart(...)

# Mismo Dataset en Framework Pytorch

In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import altair as alt

In [13]:
# Inputs: XOR problem


# Proposito: Servir como input en nuestra red y Y sera los datos reales
# Etapa: Inicializacion de matrices a utilizar

X = np.array([[0, 0, 1, 1],
              [0, 1, 0, 1]]).T  # Shape (4, 2)
y = np.array([[0, 1, 1, 0]]).T  # Shape (4, 1)



# Proposito: Convertir los datos a un vector que pueda ser utilizado por tenserflow osea un tensor
# Etapa: Inicializacion de datos


X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

# Define the model
class SimpleNet(nn.Module):
    def __init__(self):
        super(SimpleNet, self).__init__()
        # Proposito: Crear una capa completamente conectada de 2 entradas y 16 neuronas
        # Etapa: Inicializacion\
        self.fc1 = nn.Linear(2, 16)
        # Proposito: Crear una capa completamente conectada de 16 entradas y 1 salida
        # Etapa: Inicializacion
        self.fc2 = nn.Linear(16, 1)

        # Proposito: Crearla funcion de activacion de las capas
        # Etapa: Inicializacion
        self.sigmoid = nn.Sigmoid()

    # Proposito: Crearla la funcion para propagacion hacia adelante, aplicando primero la transformacion por los pesos usando la entrada y 
    # luego aplica sigboide para aplicar no linealidad. Luego lo vuelve a hacer pero con la ultima capa 
    # Etapa: Propagacion hacia adelante
    def forward(self, x):
        x = self.sigmoid(self.fc1(x))
        x = self.sigmoid(self.fc2(x))
        return x


# Proposito: Instanciar la red neuronal
# Etapa: Inicializacion

model = SimpleNet()



# Proposito: Instancia la funcion de perdida para la red neuronal
# Etapa: Inicializacion
criterion = nn.BCELoss()


# Proposito: Instancia  el optimizador de parametros
# Etapa: Inicializacion
optimizer = optim.Adam(model.parameters())

# Training loop

# Proposito: Declara un array de errores
# Etapa: Inicializacion
errors = []

# Proposito: Declara las epocas que va a usar
# Etapa: Inicializacion
epochs = 3000

for epoch in range(epochs):
    # Proposito: Lo que hacemos aqui es resetear todos los gradientes, esto lo hace  porque los acumula el modelo y si no los 
    # reseteamos lo que pasa es que estariamos aprendiendo de cosas viejas no de cosas nuevas, siendo solo memorizando informacion
    # Etapa: Retropropagacion
    optimizer.zero_grad()

    # Proposito: Lo que hacemos aqui es  obtener la salida del modelo tras hablero pasado por todas las capas
    # Etapa: Propagacion hacia adelante. 
    outputs = model(X_tensor)

    # Proposito: Lo que hacemos es obtener la prediccion de lo que se calculo con el valor real, esto dara un valor siendo nuestra perdida
    # Etapa: Calculo de perdida
    loss = criterion(outputs, y_tensor)


    # Proposito: Calcula los gradientes con respecto a los pesos y bayas. 
    # Etapa: Retropropagación
    loss.backward()

    # Proposito: Aqui es donde actualizamos los pesos conforme a su gradiente usando la funcion de
    # w_nuevo = w_actual - n * gradiente pero esto lo hace automaticamente pytorch por cada peso y cada gradiente
    # Etapa: Actualizacion de parametros
    optimizer.step()

    # Proposito: Lo unico que hace es guardar la perdida por iteracion    
    # Etapa: Graficacion
    errors.append(loss.item())

# Convert errors to DataFrame and plot
# Proposito: Lo que hace es hacer un dataset con las perdidas y las epocas por error obtenido    
# Etapa: Graficacion
df2 = pd.DataFrame({"errors": errors, "time-step": np.arange(epochs)})


# Proposito: Aqui los paso por la etapa de graficacion de las perdidas y las epocas, el punto es mostrar como baja cada vez en la curva
# Etapa: Graficacion
alt.Chart(df2).mark_line().encode(
    x="time-step",
    y="errors"
).properties(title='Chart 3')


alt.Chart(...)